# Argumind — Fine-Tuning IndoBERT (F-01) & Cross-Encoder NLI (F-02) di Google Colab

Notebook ini menjalankan pelatihan terstruktur sesuai **`research/METHODOLOGY.md`**, **`research/EXPERIMENT_PLAN.md`**, dan **`research/REPRODUCIBILITY.md`**:
1. Menghubungkan Google Colab ke repositori GitHub **`salsabilaputri95/argumind-project`** (mendukung repositori **Public** maupun **Private**).
2. Menginstal dependensi Python yang kompatibel dengan Python 3.12/3.13 Colab & memverifikasi integritas checksum `manifest.sha256`.
3. Melatih model **IndoBERT BIO Sequence Labeler** (`F-01`: `Claim`, `ExplicitPremise`, `Evidence`).
4. Melatih **Cross-Encoder NLI Relation Classifier** (`F-02`: `Support`, `Attack`, `NoRelation`) + dekode **Chu-Liu/Edmonds DAG** (`cycle_rate == 0.0`).
5. Mengekspor metrik mentah (`eval_bio.json`, `eval_nli.json`, `metrics_raw.json`) dan bobot model ke Google Drive atau unduhan lokal.

In [15]:
# @title 1. Cek GPU Google Colab & Kloning Repositori GitHub
!nvidia-smi

import os
from pathlib import Path

REPO_OWNER_AND_NAME = "salsabilaputri95/argumind-project"  # @param {type:"string"}
BRANCH = "main"  # @param {type:"string"}
GITHUB_TOKEN = ""  # @param {type:"string"}
WORK_DIR = Path("/content/argumind-project")

if not GITHUB_TOKEN.strip():
    try:
        from google.colab import userdata

        GITHUB_TOKEN = userdata.get("GITHUB_TOKEN") or ""
    except Exception:
        GITHUB_TOKEN = ""

if GITHUB_TOKEN.strip():
    CLONE_URL = f"https://{GITHUB_TOKEN.strip()}@github.com/{REPO_OWNER_AND_NAME}.git"
else:
    CLONE_URL = f"https://github.com/{REPO_OWNER_AND_NAME}.git"

if (WORK_DIR / ".git").exists():
    %cd /content/argumind-project
    !git remote set-url origin {CLONE_URL}
    !git fetch origin && git checkout {BRANCH} && git reset --hard origin/{BRANCH}
else:
    !git clone --branch {BRANCH} {CLONE_URL} /content/argumind-project

if not (WORK_DIR / ".git").exists():
    raise RuntimeError(
        "Gagal mengkloning repositori! Pastikan repo berstatus Public atau isi GITHUB_TOKEN di Sel 1."
    )

os.chdir(WORK_DIR)
!pwd
!git log -n 1 --oneline

Thu Nov 19 02:20:25 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 580.82.07              Driver Version: 580.82.07      CUDA Version: 13.0     |
+-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  Tesla T4                       Off |   00000000:00:04.0 Off |                    0 |
| N/A   39C    P8             10W /   70W |       3MiB /  15360MiB |      0%      Default |
|                                         |                        |                  N/A |
+-----------------------------------------+-----

In [16]:
# @title 2. Instal Dependensi Python (Kompatibel Python 3.12/3.13 Colab) & Atur PYTHONPATH
!pip install -q "transformers>=4.46.0" "datasets>=3.1.0" "accelerate>=1.1.0" \
  "networkx>=3.4.0" "pydantic>=2.10.0" "bert-score>=0.3.13" "krippendorff>=0.8.0" \
  python-docx pymupdf

import sys

import torch

if not WORK_DIR.exists():
    raise RuntimeError("Direktori /content/argumind-project belum ada! Pastikan Sel 1 berhasil dijalankan.")

os.chdir(WORK_DIR)

paths_to_add = []
for rel in [
    "argumind-platform/packages/shared-schemas/src",
    "argumind-platform/packages/evaluation-metrics/src",
    "argumind-platform/apps/nlp-worker/src",
    "argumind-platform/apps/api/src",
    ".",
]:
    p = str((WORK_DIR / rel).resolve())
    paths_to_add.append(p)
    if p not in sys.path:
        sys.path.insert(0, p)

os.environ["PYTHONPATH"] = ":".join(paths_to_add)
print("Working Directory:", os.getcwd())
print("PYTHONPATH:", os.environ["PYTHONPATH"])
print("PyTorch:", torch.__version__, "| CUDA Available:", torch.cuda.is_available())

Working Directory: /content/argumind-project
PYTHONPATH: /content/argumind-project/argumind-platform/packages/shared-schemas/src:/content/argumind-project/argumind-platform/packages/evaluation-metrics/src:/content/argumind-project/argumind-platform/apps/nlp-worker/src:/content/argumind-project/argumind-platform/apps/api/src:/content/argumind-project
PyTorch: 2.14.1+cu130 | CUDA Available: True


In [17]:
# @title 3. Verifikasi Checksum Dataset (manifest.sha256)
from research.runners.determinism import verify_manifest_sha256

os.chdir(WORK_DIR)
DATA_DIR = "research/data/processed/v1.0"  # @param {type:"string"}
verified_hashes = verify_manifest_sha256(WORK_DIR / DATA_DIR)
for fname, sha in verified_hashes.items():
    lines = len((WORK_DIR / DATA_DIR / fname).read_text(encoding="utf-8").splitlines())
    print(f"{fname}: OK (sha256={sha[:12]}..., lines={lines})")

documents.jsonl: OK (sha256=c403494e02a2..., lines=12)
nodes.jsonl: OK (sha256=b5feb3e7e03e..., lines=36)
edges.jsonl: OK (sha256=ddad07112860..., lines=24)
gaps.jsonl: OK (sha256=2f0448e5b2c2..., lines=12)
rubric_scores.jsonl: OK (sha256=01155dba1256..., lines=48)


In [18]:
# @title 4. Fine-Tuning IndoBERT BIO Component Extractor (F-01)
import json

from research.runners.train_indobert_bio import train_indobert_bio

os.chdir(WORK_DIR)
MODEL_NAME = "indobenchmark/indobert-base-p2"  # @param ["indobenchmark/indobert-base-p2", "indobenchmark/indobert-large-p2"]
SEED = 42  # @param [42, 100, 2026] {type:"raw"}
EPOCHS = 10  # @param {type:"integer"}
BATCH_SIZE = 16  # @param {type:"integer"}
LR = 2e-5  # @param {type:"number"}

bio_summary = train_indobert_bio(
    data_dir=WORK_DIR / DATA_DIR,
    output_dir=WORK_DIR / f"research/outputs/colab_indobert_bio_seed{SEED}",
    model_name=MODEL_NAME,
    seed=int(SEED),
    epochs=int(EPOCHS),
    batch_size=int(BATCH_SIZE),
    lr=float(LR),
)
print("=== Hasil Fine-Tuning IndoBERT BIO (F-01) ===")
print(json.dumps(bio_summary, indent=2))

config.json:   0%|          | 0.00/1.53k [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/2.00 [00:00<?, ?B/s]

vocab.txt:   0%|          | 0.00/229k [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

[transformers] You passed `num_labels=7` which is incompatible to the `id2label` map of length `5`.


pytorch_model.bin: reconstructing file:   0%|          |  0.00B /  498MB            

... (4 outputs left out)

[transformers] BertForTokenClassification LOAD REPORT from: indobenchmark/indobert-base-p2
Key                 | Status     | 
--------------------+------------+-
pooler.dense.weight | UNEXPECTED | 
pooler.dense.bias   | UNEXPECTED | 
classifier.weight   | MISSING    | 
classifier.bias     | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING:	those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.
/usr/local/lib/python3.13/dist-packages/torch/autograd/graph.py:1072: UserWarning: Memory Efficient attention defaults to a non-deterministic algorithm. To explicitly enable determinism call torch.use_deterministic_algorithms(True, warn_only=False). (Triggered internally at /__w/pytorch/pytorch/aten/src/ATen/native/transformers/cuda/attention_backward.cu:910.)
  return Variable._execution_engine.run_backward(  # Calls into the C++ engine to 

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

=== Hasil Fine-Tuning IndoBERT BIO (F-01) ===
{
  "model_name": "indobenchmark/indobert-base-p2",
  "seed": 42,
  "dev_best_macro_f1_iou50": 0.0,
  "test_macro_f1_iou50": {
    "Claim": 0.0,
    "ExplicitPremise": 0.0,
    "Evidence": 0.0,
    "macro_f1": 0.0
  },
  "checkpoint_dir": "/content/argumind-project/research/outputs/colab_indobert_bio_seed42/best_model"
}


In [19]:
# @title 5. Fine-Tuning Cross-Encoder NLI Relation Classifier + DAG Decoder (F-02)
from research.runners.train_nli_relations import train_nli_relations

os.chdir(WORK_DIR)
NLI_BASE_MODEL = "indobenchmark/indobert-base-p2"  # @param ["indobenchmark/indobert-base-p2", "MoritzLaurer/mDeBERTa-v3-base-xnli-multilingual-nli-2mil7"]
NLI_EPOCHS = 8  # @param {type:"integer"}

nli_summary = train_nli_relations(
    data_dir=WORK_DIR / DATA_DIR,
    output_dir=WORK_DIR / f"research/outputs/colab_nli_rel_seed{SEED}",
    model_name=NLI_BASE_MODEL,
    seed=int(SEED),
    epochs=int(NLI_EPOCHS),
    batch_size=int(BATCH_SIZE),
    lr=float(LR),
)
print("=== Hasil Fine-Tuning Cross-Encoder NLI + DAG (F-02) ===")
print(json.dumps(nli_summary, indent=2))

[transformers] You passed `num_labels=3` which is incompatible to the `id2label` map of length `5`.


Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

[transformers] BertForSequenceClassification LOAD REPORT from: indobenchmark/indobert-base-p2
Key               | Status  | 
------------------+---------+-
classifier.weight | MISSING | 
classifier.bias   | MISSING | 

Notes:
- MISSING:	those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/201 [00:00<?, ?it/s]

=== Hasil Fine-Tuning Cross-Encoder NLI + DAG (F-02) ===
{
  "model_name": "indobenchmark/indobert-base-p2",
  "seed": 42,
  "dev_best_relation_macro_f1": 0.5,
  "test_relation_macro_f1": {
    "Support": 1.0,
    "Attack": 0.0,
    "macro_f1": 0.5
  },
  "checkpoint_dir": "/content/argumind-project/research/outputs/colab_nli_rel_seed42/best_nli_model"
}


In [24]:
# @title 6. Jalankan Runner Evaluasi Standar E-01 & E-03b serta Tampilkan Hasil
import importlib
import json

os.chdir(WORK_DIR)
hf_path = WORK_DIR / "research/runners/hf_inference.py"
hf_path.write_text(
    hf_path.read_text(encoding="utf-8").replace(
        "segment_paragraphs_and_sentences", "segment_sentences"
    ),
    encoding="utf-8",
)

import research.runners.hf_inference
import research.runners.run_experiment

importlib.reload(research.runners.hf_inference)
importlib.reload(research.runners.run_experiment)
from research.runners.run_experiment import run_experiment

run_experiment(
    exp_id="E-01",
    arch="M1_ArgumindHybrid",
    data_dir=WORK_DIR / DATA_DIR,
    split="test",
    seed=int(SEED),
    output_dir=WORK_DIR / f"research/outputs/E-01/colab_seed{SEED}",
    bio_model_dir=WORK_DIR / f"research/outputs/colab_indobert_bio_seed{SEED}/best_model",
    nli_model_dir=WORK_DIR / f"research/outputs/colab_nli_rel_seed{SEED}/best_nli_model",
)

run_experiment(
    exp_id="E-03b",
    arch="M3_GraphGroundedScoring",
    data_dir=WORK_DIR / DATA_DIR,
    split="test",
    seed=int(SEED),
    output_dir=WORK_DIR / f"research/outputs/E-03b/colab_seed{SEED}",
)

for path in [
    f"research/outputs/colab_indobert_bio_seed{SEED}/eval_bio.json",
    f"research/outputs/colab_nli_rel_seed{SEED}/eval_nli.json",
    f"research/outputs/E-01/colab_seed{SEED}/metrics_raw.json",
    f"research/outputs/E-03b/colab_seed{SEED}/metrics_raw.json",
]:
    p = WORK_DIR / path
    if p.exists():
        print(f"=== {path} ===")
        print(json.dumps(json.loads(p.read_text(encoding="utf-8")), indent=2))
    else:
        print(f"[WARNING] File belum ditemukan: {path}")

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/201 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/201 [00:00<?, ?it/s]

=== research/outputs/colab_indobert_bio_seed42/eval_bio.json ===
{
  "model_name": "indobenchmark/indobert-base-p2",
  "seed": 42,
  "dev_best_macro_f1_iou50": 0.0,
  "test_macro_f1_iou50": {
    "Claim": 0.0,
    "ExplicitPremise": 0.0,
    "Evidence": 0.0,
    "macro_f1": 0.0
  },
  "checkpoint_dir": "/content/argumind-project/research/outputs/colab_indobert_bio_seed42/best_model"
}
=== research/outputs/colab_nli_rel_seed42/eval_nli.json ===
{
  "model_name": "indobenchmark/indobert-base-p2",
  "seed": 42,
  "dev_best_relation_macro_f1": 0.5,
  "test_relation_macro_f1": {
    "Support": 1.0,
    "Attack": 0.0,
    "macro_f1": 0.5
  },
  "checkpoint_dir": "/content/argumind-project/research/outputs/colab_nli_rel_seed42/best_nli_model"
}
=== research/outputs/E-01/colab_seed42/metrics_raw.json ===
{
  "component_macro_f1_exact": 0.0,
  "component_macro_f1_iou50": 0.0,
  "relation_macro_f1": 0.0,
  "cycle_rate": 0.0,
  "n_docs": 2,
  "latency_total_ms": 2334.9
}
=== research/outputs/E-03

In [26]:
# @title 7. Simpan Checkpoint & Artefak ke Google Drive / Unduh ZIP
import shutil

os.chdir(WORK_DIR)
archive_base = WORK_DIR / f"argumind_colab_outputs_seed{SEED}"
zip_file = Path(shutil.make_archive(str(archive_base), "zip", root_dir=WORK_DIR, base_dir="research/outputs"))
print("Arsip siap:", zip_file, "| Exists:", zip_file.exists(), "| Size (bytes):", zip_file.stat().st_size)

SAVE_TO_DRIVE = True  # @param {type:"boolean"}
if SAVE_TO_DRIVE and zip_file.exists():
    from google.colab import drive

    drive.mount("/content/drive")
    drive_dir = Path("/content/drive/MyDrive/argumind_outputs")
    drive_dir.mkdir(parents=True, exist_ok=True)
    shutil.copy2(zip_file, drive_dir / zip_file.name)
    print("Tersalin ke Google Drive:", drive_dir / zip_file.name)

Arsip siap: /content/argumind-project/argumind_colab_outputs_seed42.zip | Exists: True | Size (bytes): 922417466
Mounted at /content/drive
Tersalin ke Google Drive: /content/drive/MyDrive/argumind_outputs/argumind_colab_outputs_seed42.zip
